# 🔬 CDLib — Kaggle Training Driver

**Zero model/loss/data logic here.** This notebook only:
1. Bootstraps the environment (clone, install)
2. Sets up W&B key from Kaggle Secrets
3. Runs training via CLI
4. Plots results

All logic lives in the `cdlib` package.

### Kaggle Notes
- Results are saved to `/kaggle/working/` (persists across saves)
- Use **Add-ons → Secrets** to store `WANDB_API_KEY`
- GPU sessions last ~12h (vs Colab's ~90min free tier)

## Cell 1: Bootstrap

In [ ]:
# Clone repo and install
!git clone https://github.com/Deep-NeuralNetworks-Research-Project/baseline-infa-bula.git /kaggle/working/cdlib-repo 2>/dev/null || (cd /kaggle/working/cdlib-repo && git pull)
%cd /kaggle/working/cdlib-repo
!pip install -e . -q

## Cell 2: W&B Login (via Kaggle Secrets)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('✅ W&B key loaded from Kaggle Secrets.')
except Exception:
    print('⚠️  W&B key not found. Add it via Add-ons → Secrets.')
    print('   Training will continue without W&B logging.')

## Cell 3: Train (one command)

In [ ]:
# Change the experiment config as needed:
#   +experiment=baseline_fcsiamdiff_sysu
#   +experiment=ablation_no_alignment
#   resume_from=auto  (to continue after a restart)

!python -m cdlib.cli.train +experiment=baseline_fcsiamdiff_sysu

## Cell 4: Resume after restart

In [ ]:
# If Kaggle session expired, re-run cells 1-2, then run this:
# !python -m cdlib.cli.train +experiment=baseline_fcsiamdiff_sysu train.checkpoint.resume_from=auto

## Cell 5: Plot Results

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# Find the latest run directory
results_dir = Path('results')
if results_dir.exists():
    runs = sorted(results_dir.iterdir())
    if runs:
        latest_run = runs[-1]
        print(f'Latest run: {latest_run}')
        print(f'Contents: {list(latest_run.iterdir())}')
    else:
        print('No runs found yet.')
else:
    print('results/ directory not found. Run training first.')